# Vigilance Decrement

> **Task name:** `Vigilance Decrement`

**Track: Attention** | Does accuracy decay over a long sequence of repeated sub-tasks?

In sustained attention research, humans show a reliable "vigilance decrement" —
performance degrades over time on monotonous monitoring tasks. This benchmark tests
whether LLMs show analogous decay by presenting 100 identical sub-tasks in a single
prompt and measuring accuracy as a function of position.

**Metrics:**
- Accuracy curve across sequence positions
- Decay onset (position where accuracy drops below 95%)
- Overall accuracy and standard deviation across positions

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(response: str) -> str:
    if "</think>" in response:
        return response.split("</think>")[-1].strip()
    return response

def parse_numbered_answers(response: str, count: int) -> list[str]:
    response = strip_thinking(response)
    lines = response.strip().split("\n")
    answers = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if re.match(r"^\d+[\.)\:\-]", line):
            cleaned = re.sub(r"^\d+[\.)\:\-]\s*", "", line)
            if cleaned:
                answers.append(cleaned)
    if not answers:
        for line in lines:
            line = line.strip()
            if line:
                answers.append(line)
    while len(answers) < count:
        answers.append("")
    return answers[:count]

def check_answer(expected: str, actual: str) -> bool:
    exp = expected.lower().strip()
    act = actual.lower().strip()
    if exp == act or exp in act:
        return True
    exp_core = re.sub(r"^(approximately|about|roughly|around)\s+", "", exp)
    if exp_core in act:
        return True
    exp_d = re.sub(r"[,\s]", "", exp)
    act_d = re.sub(r"[,\s]", "", act)
    return exp_d == act_d and exp_d != ""

In [ ]:
# (city, country) pairs for country identification
CITY_COUNTRY_PAIRS = [
    ("Tokyo", "Japan"), ("Paris", "France"), ("Cairo", "Egypt"),
    ("Buenos Aires", "Argentina"), ("Stockholm", "Sweden"),
    ("Lagos", "Nigeria"), ("Mumbai", "India"), ("Seoul", "South Korea"),
    ("Lima", "Peru"), ("Warsaw", "Poland"),
    ("Nairobi", "Kenya"), ("Bangkok", "Thailand"), ("Lisbon", "Portugal"),
    ("Bogotá", "Colombia"), ("Hanoi", "Vietnam"),
    ("Prague", "Czech Republic"), ("Santiago", "Chile"), ("Accra", "Ghana"),
    ("Bucharest", "Romania"), ("Dhaka", "Bangladesh"),
    ("Oslo", "Norway"), ("Havana", "Cuba"), ("Jakarta", "Indonesia"),
    ("Athens", "Greece"), ("Ankara", "Turkey"),
    ("Riyadh", "Saudi Arabia"), ("Dublin", "Ireland"), ("Quito", "Ecuador"),
    ("Helsinki", "Finland"), ("Manila", "Philippines"),
    ("Tunis", "Tunisia"), ("Montevideo", "Uruguay"), ("Bratislava", "Slovakia"),
    ("Amman", "Jordan"), ("Tbilisi", "Georgia"),
    ("Canberra", "Australia"), ("Ottawa", "Canada"), ("Brasília", "Brazil"),
    ("Wellington", "New Zealand"), ("Bern", "Switzerland"),
    ("Dakar", "Senegal"), ("Kuala Lumpur", "Malaysia"), ("Colombo", "Sri Lanka"),
    ("Addis Ababa", "Ethiopia"), ("Tashkent", "Uzbekistan"),
    ("Reykjavik", "Iceland"), ("Tallinn", "Estonia"), ("Riga", "Latvia"),
    ("Vilnius", "Lithuania"), ("Ljubljana", "Slovenia"),
    ("Zagreb", "Croatia"), ("Sarajevo", "Bosnia and Herzegovina"),
    ("Maputo", "Mozambique"), ("Lusaka", "Zambia"), ("Kampala", "Uganda"),
    ("Muscat", "Oman"), ("Doha", "Qatar"), ("Kuwait City", "Kuwait"),
    ("Windhoek", "Namibia"), ("Gaborone", "Botswana"),
    ("Yerevan", "Armenia"), ("Baku", "Azerbaijan"), ("Ulaanbaatar", "Mongolia"),
    ("Phnom Penh", "Cambodia"), ("Vientiane", "Laos"),
    ("Kathmandu", "Nepal"), ("Islamabad", "Pakistan"), ("Kabul", "Afghanistan"),
    ("Minsk", "Belarus"), ("Chisinau", "Moldova"),
    ("Skopje", "North Macedonia"), ("Tirana", "Albania"), ("Podgorica", "Montenegro"),
    ("Belgrade", "Serbia"), ("Sofia", "Bulgaria"),
    ("Nicosia", "Cyprus"), ("Valletta", "Malta"), ("Luxembourg City", "Luxembourg"),
    ("San José", "Costa Rica"), ("Panama City", "Panama"),
    ("Kingston", "Jamaica"), ("Port-au-Prince", "Haiti"),
    ("Tegucigalpa", "Honduras"), ("Managua", "Nicaragua"),
    ("Guatemala City", "Guatemala"), ("San Salvador", "El Salvador"),
    ("Rabat", "Morocco"), ("Algiers", "Algeria"),
    ("Tripoli", "Libya"), ("Khartoum", "Sudan"),
    ("Dar es Salaam", "Tanzania"), ("Harare", "Zimbabwe"),
    ("Antananarivo", "Madagascar"), ("Bamako", "Mali"),
    ("Ouagadougou", "Burkina Faso"), ("Niamey", "Niger"),
    ("N'Djamena", "Chad"), ("Bangui", "Central African Republic"),
    ("Libreville", "Gabon"), ("Brazzaville", "Republic of the Congo"),
    ("Kinshasa", "Democratic Republic of the Congo"),
    ("Luanda", "Angola"), ("Lilongwe", "Malawi"), ("Asmara", "Eritrea"),
]

COUNTRY_TEMPLATES = [
    "The international conference held in {city} attracted delegates from across the region.",
    "Researchers at the national university in {city} published their findings on urban development.",
    "The new metro line in {city} was officially inaugurated after three years of construction.",
    "A delegation from {city} arrived at the summit to discuss renewable energy policies.",
    "The historic district of {city} was recently added to the UNESCO World Heritage list.",
    "The annual literary festival in {city} featured over 200 authors from five continents.",
    "Exports from the industrial zone near {city} increased by 12 percent over the previous year.",
    "The national museum in {city} unveiled a collection of artifacts dating back 3,000 years.",
    "Public transportation upgrades in {city} reduced average commute times by 25 percent.",
    "The botanical garden in {city} houses over 4,000 plant species from tropical regions.",
    "A new desalination plant outside {city} will supply drinking water to 500,000 residents.",
    "The central bank headquartered in {city} announced a reduction in interest rates.",
    "Medical researchers at {city}'s main hospital developed a novel treatment for malaria.",
    "The port authority in {city} processed a record 2.3 million shipping containers last quarter.",
    "Architects in {city} completed a solar-powered housing complex for 1,200 families.",
    "The technology startup scene in {city} has attracted over $800 million in venture capital.",
    "Environmental activists in {city} organized a cleanup of the river running through the capital.",
    "The opera house in {city} reopened after an 18-month renovation costing $45 million.",
    "A rare manuscript was discovered in the national archives of {city} during routine cataloguing.",
    "The annual marathon through the streets of {city} drew 42,000 runners this year.",
]

# Fake city names for target-absent country identification trials
FAKE_CITIES = [
    "Porthaven", "Zelindra", "Novaris", "Caltheon", "Bryndova",
    "Orenthal", "Kelviston", "Suranta", "Melvora", "Triandos",
    "Veldmark", "Quorath", "Hestiga", "Dranthor", "Yelvina",
    "Pellucra", "Wynthara", "Gormunde", "Fessalia", "Obrantis",
]

# Number extraction seed data
NUMBER_CONTEXTS = [
    ("The warehouse stored exactly {n} crates of medical supplies for distribution.", "crates"),
    ("The survey recorded {n} species of migratory birds in the wetland reserve.", "species"),
    ("Engineers installed {n} solar panels on the roof of the convention center.", "panels"),
    ("The orchard produced {n} tonnes of apples during the autumn harvest.", "tonnes"),
    ("Archaeologists catalogued {n} pottery fragments from the excavation site.", "fragments"),
    ("The library acquired {n} rare manuscripts at the annual auction.", "manuscripts"),
    ("Factory output reached {n} units per day after the efficiency upgrade.", "units"),
    ("The census counted {n} households in the newly incorporated township.", "households"),
    ("Volunteers planted {n} trees along the riverbank restoration project.", "trees"),
    ("The telescope array detected signals from {n} previously unknown pulsars.", "pulsars"),
    ("The ferry transported {n} passengers across the strait during peak season.", "passengers"),
    ("Inspectors found {n} structural defects during the bridge safety audit.", "defects"),
    ("The vineyard bottled {n} cases of reserve wine from the 2018 vintage.", "cases"),
    ("Scientists tagged {n} sea turtles during the nesting season survey.", "turtles"),
    ("The power station generated {n} megawatt-hours of electricity last month.", "megawatt-hours"),
    ("The museum welcomed {n} visitors during its opening weekend.", "visitors"),
    ("Workers laid {n} meters of fiber optic cable through the city center.", "meters"),
    ("The farm raised {n} head of cattle on its 500-hectare property.", "head"),
    ("The competition attracted {n} entries from amateur photographers.", "entries"),
    ("Geologists drilled {n} core samples from the proposed dam site.", "samples"),
]

# Templates for number-absent trials (no number present)
NUMBER_ABSENT_TEMPLATES = [
    "The delegation reviewed the environmental impact report before approving the project.",
    "Researchers presented their findings on coral reef degradation at the symposium.",
    "The committee postponed the vote on the infrastructure proposal until next quarter.",
    "Engineers completed the feasibility study for the proposed highway extension.",
    "The ministry issued new guidelines for wastewater treatment in coastal regions.",
    "Inspectors verified compliance with fire safety regulations across all facilities.",
    "The research team published their conclusions on soil erosion in the journal.",
    "Architects submitted revised blueprints for the cultural center renovation.",
    "The board approved the merger after reviewing the due diligence report.",
    "Scientists confirmed the presence of rare minerals in the geological survey.",
    "The agency released its annual assessment of air quality in metropolitan areas.",
    "Volunteers coordinated the distribution of relief supplies to affected communities.",
    "The consortium signed the agreement for the cross-border pipeline project.",
    "Auditors completed their review of the procurement process for irregularities.",
    "The university established a new research center for renewable energy studies.",
]

# Misspelling seed data: (correct_word, misspelled_word)
MISSPELLING_PAIRS = [
    ("examined", "exmained"), ("necessary", "neccessary"), ("occurred", "occured"),
    ("separate", "seperate"), ("definitely", "definately"), ("environment", "enviroment"),
    ("temperature", "temprature"), ("laboratory", "labratory"), ("restaurant", "restaraunt"),
    ("government", "goverment"), ("beginning", "begining"), ("immediately", "immediatley"),
    ("professional", "proffesional"), ("particularly", "particularily"), ("independent", "independant"),
    ("apparently", "apparantly"), ("equipment", "equiptment"), ("experience", "experiance"),
    ("maintenance", "maintainance"), ("technique", "tecnique"), ("sufficient", "sufficent"),
    ("accommodate", "accomodate"), ("committee", "commitee"), ("development", "developement"),
    ("approximately", "approximatley"), ("concerned", "concerened"), ("knowledge", "knowlege"),
    ("conscience", "concience"), ("guarantee", "guarentee"), ("colleague", "colleauge"),
    ("permanent", "permenent"), ("privilege", "privelege"), ("questionnaire", "questionaire"),
    ("rhythm", "rythm"), ("schedule", "scedule"), ("surveillance", "surveilance"),
    ("threshold", "threshhold"), ("vulnerable", "vunerable"), ("achievement", "acheivment"),
    ("appreciate", "apprecaite"), ("disappear", "dissapear"), ("calendar", "calender"),
    ("category", "catagory"), ("cemetery", "cemetary"), ("consensus", "concensus"),
    ("desperate", "desparate"), ("embarrass", "embarass"), ("fascinate", "facinate"),
    ("harass", "harrass"), ("intelligence", "intelligance"), ("lieutenant", "leiutenant"),
    ("millennium", "millenium"), ("noticeable", "noticable"),
]

MISSPELLING_TEMPLATES = [
    "The professor carefully {word} the ancient manuscript under ultraviolet light.",
    "It was {word} to complete all forms before the submission deadline.",
    "The incident {word} during the early morning hours when few staff were present.",
    "The committee voted to {word} the budget into three distinct allocations.",
    "The results were {word} better than any previous attempt at the procedure.",
    "The {word} impact of the new regulation surprised industry analysts.",
    "The {word} reading was taken at dawn before the chamber was opened.",
    "Samples were processed in the {word} under strict contamination protocols.",
    "The group decided to meet at the {word} near the conference center.",
    "The {word} issued a statement regarding the new infrastructure project.",
    "The ceremony marked the {word} of a new era for the institution.",
    "The team responded {word} to the emergency alert from the monitoring station.",
    "The {word} association awarded its annual prize to the research team.",
    "The study focused {word} on the effects of urbanization on bird populations.",
    "The board declared the organization fully {word} of external funding sources.",
    "The decline was {word} visible in the quarterly financial reports.",
    "All {word} was inspected before being deployed to the field stations.",
    "Years of {word} in tropical fieldwork prepared her for the expedition.",
    "Regular {word} of the filtration system prevented costly breakdowns.",
    "The new {word} significantly reduced processing time in the assembly line.",
]

# Templates for misspelling-absent trials (all words correctly spelled)
MISSPELLING_ABSENT_TEMPLATES = [
    "The committee reviewed the proposal and approved the funding allocation unanimously.",
    "Researchers published their findings on atmospheric pressure in the quarterly journal.",
    "The delegation arrived at the conference center before the opening ceremony began.",
    "Engineers completed the structural assessment of the bridge ahead of schedule.",
    "The museum curator arranged the exhibition according to the chronological timeline.",
    "Volunteers distributed educational materials to students in the rural district.",
    "The observatory recorded unusual solar activity during the autumn equinox period.",
    "Architects presented their design for the sustainable housing development project.",
    "The agricultural department released updated guidelines for irrigation management.",
    "Scientists identified a previously unknown species of beetle in the forest reserve.",
    "The transport authority announced improvements to the regional bus network.",
    "Auditors confirmed that all financial records were accurate and complete.",
    "The hospital expanded its emergency department to accommodate increased demand.",
    "The university awarded scholarships to outstanding students from disadvantaged backgrounds.",
    "The environmental agency monitored water quality throughout the monsoon season.",
]

print(f"Seed data loaded: {len(CITY_COUNTRY_PAIRS)} city-country pairs, {len(COUNTRY_TEMPLATES)} templates")
print(f"  {len(FAKE_CITIES)} fake cities for target-absent trials")
print(f"  {len(NUMBER_CONTEXTS)} number contexts, {len(NUMBER_ABSENT_TEMPLATES)} number-absent templates")
print(f"  {len(MISSPELLING_PAIRS)} misspelling pairs, {len(MISSPELLING_TEMPLATES)} misspelling templates")
print(f"  {len(MISSPELLING_ABSENT_TEMPLATES)} misspelling-absent templates")

In [ ]:
def generate_vigilance_country(count: int, rng: random.Random,
                                oddball_position: int | None = None,
                                absent_positions: list[int] | None = None) -> dict:
    """Generate a country-identification vigilance task with optional target-absent trials."""
    absent_positions = absent_positions or []
    pairs = rng.sample(CITY_COUNTRY_PAIRS, min(count, len(CITY_COUNTRY_PAIRS)))
    while len(pairs) < count:
        pairs.extend(rng.sample(CITY_COUNTRY_PAIRS, min(count - len(pairs), len(CITY_COUNTRY_PAIRS))))

    templates = COUNTRY_TEMPLATES[:]
    items_text = []
    answers = []
    is_target_present = []  # For SDT: True = target present, False = target absent

    for i, (city, country) in enumerate(pairs):
        template = rng.choice(templates)
        if oddball_position is not None and i == oddball_position:
            sentence = f"The warehouse in {city} stored exactly 4,731 crates of supplies for distribution."
            items_text.append(f"Item {i+1}: {sentence}")
            answers.append("4,731")
            is_target_present.append(True)
        elif i in absent_positions:
            # Target-absent: use a fake city name — no valid country
            fake_city = rng.choice(FAKE_CITIES)
            sentence = template.format(city=fake_city)
            items_text.append(f"Item {i+1}: {sentence}")
            answers.append("UNKNOWN")
            is_target_present.append(False)
        else:
            sentence = template.format(city=city)
            items_text.append(f"Item {i+1}: {sentence}")
            answers.append(country)
            is_target_present.append(True)

    oddball_note = ""
    if oddball_position is not None:
        oddball_note = (
            "\n\nNote: If any item asks for something other than a country, "
            "provide the appropriate answer for that item instead."
        )

    absent_note = ""
    if absent_positions:
        absent_note = (
            "\n\nNote: If a city name is not real or you cannot identify the country, "
            "respond with exactly: UNKNOWN"
        )

    prompt = (
        f"For each of the following {count} items, identify which country is being "
        "described or referenced. Respond with ONLY the country name for each item, "
        f"one per line, numbered to match.{oddball_note}{absent_note}\n\n"
        + "\n".join(items_text)
        + "\n\nAnswers:"
    )
    return {"prompt": prompt, "answers": answers, "is_target_present": is_target_present}


def generate_vigilance_numbers(count: int, rng: random.Random,
                                oddball_position: int | None = None,
                                absent_positions: list[int] | None = None) -> dict:
    """Generate a number-extraction vigilance task with optional target-absent trials."""
    absent_positions = absent_positions or []
    items_text = []
    answers = []
    is_target_present = []

    for i in range(count):
        if oddball_position is not None and i == oddball_position:
            city, country = rng.choice(CITY_COUNTRY_PAIRS)
            sentence = f"The delegation from {city} arrived to inspect the facility."
            items_text.append(f"Item {i+1}: {sentence}")
            answers.append(country)
            is_target_present.append(True)
        elif i in absent_positions:
            # Target-absent: sentence with no number
            sentence = rng.choice(NUMBER_ABSENT_TEMPLATES)
            items_text.append(f"Item {i+1}: {sentence}")
            answers.append("NONE")
            is_target_present.append(False)
        else:
            ctx_template, _ = rng.choice(NUMBER_CONTEXTS)
            n = rng.randint(10, 99999)
            n_str = f"{n:,}"
            sentence = ctx_template.format(n=n_str)
            items_text.append(f"Item {i+1}: {sentence}")
            answers.append(n_str)
            is_target_present.append(True)

    oddball_note = ""
    if oddball_position is not None:
        oddball_note = (
            "\n\nNote: If any item does not contain a number, identify the country "
            "referenced instead."
        )

    absent_note = ""
    if absent_positions:
        absent_note = (
            "\n\nNote: If a sentence contains no number, respond with exactly: NONE"
        )

    prompt = (
        f"For each of the following {count} items, extract the number mentioned. "
        "Respond with ONLY the number for each item, one per line, numbered to match."
        f"{oddball_note}{absent_note}\n\n"
        + "\n".join(items_text)
        + "\n\nAnswers:"
    )
    return {"prompt": prompt, "answers": answers, "is_target_present": is_target_present}


def generate_vigilance_misspelling(count: int, rng: random.Random,
                                    oddball_position: int | None = None,
                                    absent_positions: list[int] | None = None) -> dict:
    """Generate a misspelling-detection vigilance task with optional target-absent trials."""
    absent_positions = absent_positions or []
    items_text = []
    answers = []
    is_target_present = []

    for i in range(count):
        if oddball_position is not None and i == oddball_position:
            n = rng.randint(100, 9999)
            sentence = f"The facility processed exactly {n:,} samples during the quarterly review."
            items_text.append(f"Item {i+1}: {sentence}")
            answers.append(f"{n:,}")
            is_target_present.append(True)
        elif i in absent_positions:
            # Target-absent: correctly spelled sentence
            sentence = rng.choice(MISSPELLING_ABSENT_TEMPLATES)
            items_text.append(f"Item {i+1}: {sentence}")
            answers.append("NONE")
            is_target_present.append(False)
        else:
            correct, misspelled = rng.choice(MISSPELLING_PAIRS)
            template = rng.choice(MISSPELLING_TEMPLATES)
            sentence = template.format(word=misspelled)
            items_text.append(f"Item {i+1}: {sentence}")
            answers.append(misspelled)
            is_target_present.append(True)

    oddball_note = ""
    if oddball_position is not None:
        oddball_note = (
            "\n\nNote: If any item does not contain a misspelling, extract the "
            "number mentioned instead."
        )

    absent_note = ""
    if absent_positions:
        absent_note = (
            "\n\nNote: If a sentence has no misspelling, respond with exactly: NONE"
        )

    prompt = (
        f"For each of the following {count} items, identify the misspelled word. "
        "Respond with ONLY the misspelled word for each item, one per line, "
        f"numbered to match.{oddball_note}{absent_note}\n\n"
        + "\n".join(items_text)
        + "\n\nAnswers:"
    )
    return {"prompt": prompt, "answers": answers, "is_target_present": is_target_present}


# --- Generate dataset ---
rng = random.Random(123)
COUNT = 100
data = []

generators = {
    "country_identification": generate_vigilance_country,
    "number_extraction": generate_vigilance_numbers,
    "misspelling_detection": generate_vigilance_misspelling,
}

for task_type, gen_fn in generators.items():
    # Normal variant (no absent trials)
    result = gen_fn(COUNT, rng)
    data.append({
        "task_id": f"vig_{task_type}_normal",
        "task_type": task_type,
        "variant": "normal",
        "num_subtasks": COUNT,
        "prompt": result["prompt"],
        "answers": result["answers"],
        "is_target_present": result["is_target_present"],
        "oddball_position": None,
    })

    # Oddball variant
    oddball_pos = rng.randint(40, 70)
    result = gen_fn(COUNT, rng, oddball_position=oddball_pos)
    data.append({
        "task_id": f"vig_{task_type}_oddball",
        "task_type": task_type,
        "variant": "oddball",
        "num_subtasks": COUNT,
        "prompt": result["prompt"],
        "answers": result["answers"],
        "is_target_present": result["is_target_present"],
        "oddball_position": oddball_pos,
    })

    # SDT variant: ~15% target-absent trials at random positions
    absent_pos = sorted(rng.sample(range(COUNT), 15))
    result = gen_fn(COUNT, rng, absent_positions=absent_pos)
    data.append({
        "task_id": f"vig_{task_type}_sdt",
        "task_type": task_type,
        "variant": "sdt",
        "num_subtasks": COUNT,
        "prompt": result["prompt"],
        "answers": result["answers"],
        "is_target_present": result["is_target_present"],
        "oddball_position": None,
    })

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} items")
print(f"By task type: {dict(df_all['task_type'].value_counts())}")
print(f"By variant: {dict(df_all['variant'].value_counts())}")

## Task Definition

In [ ]:
@kbench.task(
    name="vigilance_decrement",
    description="Does accuracy decay over 100 repeated trivially-easy sub-tasks in a single prompt?"
)
def vigilance_decrement(llm, prompt: str, answers: str, num_subtasks: int) -> tuple[int, int]:
    response = llm.prompt(prompt)
    expected = json.loads(answers) if isinstance(answers, str) else answers
    num_sub = int(num_subtasks)
    parsed = parse_numbered_answers(response, num_sub)
    correct = sum(1 for e, a in zip(expected, parsed) if check_answer(e, a))
    return (correct, num_sub)

## Run Evaluation

Uses `kbench.llm` — the default model. Use Kaggle's **"Add Models"** button to run across multiple models.

In [ ]:
eval_df = df_all.copy()
eval_df["answers"] = eval_df["answers"].apply(json.dumps)
eval_df["is_target_present"] = eval_df["is_target_present"].apply(json.dumps)
task_eval_df = eval_df[["prompt", "answers", "num_subtasks"]].copy()

print(f"Running {len(task_eval_df)} vigilance items with kbench.llm...")
runs = vigilance_decrement.evaluate(
    llm=[kbench.llm],
    evaluation_data=task_eval_df,
    n_jobs=2,
)
results = runs.as_dataframe()
results["accuracy"] = results["result"].apply(
    lambda r: r[0] / r[1] if isinstance(r, tuple) and r[1] > 0 else float(r) if not isinstance(r, tuple) else 0
)
results = results.merge(
    eval_df[["prompt", "task_type", "variant", "task_id", "oddball_position", "is_target_present", "answers"]],
    on="prompt", how="left"
)
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
print("=== Vigilance Results ===")
print(results.groupby(["task_type", "variant"])["accuracy"].mean().to_string(float_format="%.3f"))

print("\n=== Oddball Detection ===")
for _, row in results[results["variant"] == "oddball"].iterrows():
    tt = row["task_type"]
    acc = row["accuracy"]
    print(f"  {tt}: overall accuracy = {acc:.2%}")

## Signal Detection Theory (SDT) Analysis

For the SDT variants, we decompose performance into **sensitivity** (d') and **criterion** (c) across position windows to separate the model's ability to detect targets from its response bias.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy", "matplotlib"], check=True)
from scipy.stats import norm
import matplotlib.pyplot as plt

def compute_dprime(hit_rate: float, fa_rate: float) -> tuple[float, float]:
    """Compute d' (sensitivity) and c (criterion) from hit and false alarm rates."""
    # Clamp to avoid infinite Z-scores
    hr = np.clip(hit_rate, 0.01, 0.99)
    far = np.clip(fa_rate, 0.01, 0.99)
    d_prime = norm.ppf(hr) - norm.ppf(far)
    criterion = -0.5 * (norm.ppf(hr) + norm.ppf(far))
    return d_prime, criterion

# Analyze SDT variants only
sdt_results = results[results["variant"] == "sdt"].copy()

if len(sdt_results) > 0:
    print("=== SDT Analysis by Position Window ===\n")
    
    WINDOW_SIZE = 10
    all_sdt_data = []
    
    for _, row in sdt_results.iterrows():
        task_type = row["task_type"]
        answers_expected = json.loads(row["answers"]) if isinstance(row["answers"], str) else row["answers"]
        is_present = json.loads(row["is_target_present"]) if isinstance(row["is_target_present"], str) else row["is_target_present"]
        
        # Get model's parsed responses
        response = row.get("response", "")
        if not response and "result" in row:
            # We need to re-parse — use the stored result
            result_tuple = row["result"]
            correct_count = result_tuple[0] if isinstance(result_tuple, tuple) else 0
        
        # Parse per-item results from the stored answers
        parsed = parse_numbered_answers(str(row.get("response", "")), len(answers_expected))
        
        for window_start in range(0, len(answers_expected), WINDOW_SIZE):
            window_end = min(window_start + WINDOW_SIZE, len(answers_expected))
            window_label = f"{window_start+1}-{window_end}"
            
            hits = 0
            misses = 0
            false_alarms = 0
            correct_rejections = 0
            
            for j in range(window_start, window_end):
                if j >= len(parsed) or j >= len(answers_expected):
                    break
                
                expected = answers_expected[j]
                actual = parsed[j] if j < len(parsed) else ""
                present = is_present[j]
                correct = check_answer(expected, actual)
                
                if present:
                    if correct:
                        hits += 1
                    else:
                        misses += 1
                else:
                    # Target absent — check if model correctly said UNKNOWN/NONE
                    absent_markers = ["unknown", "none", "n/a", "no country", "not real"]
                    model_says_absent = any(m in actual.lower() for m in absent_markers)
                    if model_says_absent:
                        correct_rejections += 1
                    else:
                        false_alarms += 1
            
            total_present = hits + misses
            total_absent = false_alarms + correct_rejections
            
            hr = hits / total_present if total_present > 0 else 0.5
            far = false_alarms / total_absent if total_absent > 0 else 0.5
            
            d_prime, criterion = compute_dprime(hr, far)
            
            all_sdt_data.append({
                "task_type": task_type,
                "window": window_label,
                "window_start": window_start,
                "hit_rate": hr,
                "false_alarm_rate": far,
                "d_prime": d_prime,
                "criterion": criterion,
                "n_present": total_present,
                "n_absent": total_absent,
            })
    
    sdt_df = pd.DataFrame(all_sdt_data)
    
    print("Position Window | Hit Rate | FA Rate |   d'   | Criterion")
    print("-" * 60)
    for _, r in sdt_df.groupby("window_start").agg({
        "hit_rate": "mean", "false_alarm_rate": "mean",
        "d_prime": "mean", "criterion": "mean"
    }).iterrows():
        wl = f"{_+1}-{_+WINDOW_SIZE}"
        print(f"  {wl:>12}   |  {r['hit_rate']:.3f}  | {r['false_alarm_rate']:.3f}  | {r['d_prime']:+.3f} | {r['criterion']:+.3f}")
    
    # Plot d' by position window
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    for tt in sdt_df["task_type"].unique():
        subset = sdt_df[sdt_df["task_type"] == tt].sort_values("window_start")
        ax1.plot(subset["window_start"] + WINDOW_SIZE/2, subset["d_prime"], 
                "o-", label=tt.replace("_", " ").title(), linewidth=2, markersize=6)
    
    ax1.set_xlabel("Position in Sequence")
    ax1.set_ylabel("Sensitivity (d')")
    ax1.set_title("Vigilance Decrement: SDT Sensitivity by Position")
    ax1.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
    ax1.legend()
    ax1.set_xticks(range(5, 100, 10))
    ax1.set_xticklabels([f"{i}-{i+9}" for i in range(1, 100, 10)], rotation=45, fontsize=8)
    
    for tt in sdt_df["task_type"].unique():
        subset = sdt_df[sdt_df["task_type"] == tt].sort_values("window_start")
        ax2.plot(subset["window_start"] + WINDOW_SIZE/2, subset["criterion"],
                "s--", label=tt.replace("_", " ").title(), linewidth=2, markersize=6)
    
    ax2.set_xlabel("Position in Sequence")
    ax2.set_ylabel("Criterion (c)")
    ax2.set_title("Response Bias Shift Over Sequence")
    ax2.axhline(y=0, color="gray", linestyle="--", alpha=0.3, label="No bias")
    ax2.legend()
    ax2.set_xticks(range(5, 100, 10))
    ax2.set_xticklabels([f"{i}-{i+9}" for i in range(1, 100, 10)], rotation=45, fontsize=8)
    
    plt.tight_layout()
    plt.savefig("vigilance_sdt.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("SDT plot saved to vigilance_sdt.png")
else:
    print("No SDT variant results available")